## LangChain聊天机器人

In [1]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, SystemMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import HumanMessage, AIMessage, SystemMessage

In [2]:
# 定义模型
# model_name = 'qwen2.5:1.5b'
model_name = 'qwen2.5:7b'
ollama_base_url = os.environ.get("OLLAMA_API_BASE")

model = init_chat_model(model=model_name, model_provider="ollama", base_url=ollama_base_url)

In [3]:
# str parser
str_parser = StrOutputParser()

chain = model | str_parser

### 1. 简单对话，无记忆

In [4]:
# 告诉模型我的名字
messages = [
    HumanMessage(content="Hi! I am alex.")
]
chain.invoke(messages)

'Hello Alex! Nice to meet you. How can I assist you today? Whether you have questions, need some information, or just want to chat, feel free to let me know!'

In [5]:
# 直接问model我的名字，它会不知我的名字
chain.invoke([HumanMessage(content="What's my name?")])

"You didn't provide your name in your question. In our conversation, you only asked about the name, so I don't have enough information to answer accurately. If you tell me your name, I'd be happy to confirm it for you!"

**模型不知道我们的名字**

### 2. 携带历史消息会话

In [6]:
# 携带历史消息
messages = [
    HumanMessage(content="Hi! I am alex."),
    AIMessage(content="Hello Alex! It's nice to meet you. How can I assist you today?"),
    HumanMessage(content="What's my name?")
]
# 会话
chain.invoke(messages)

"Your name is Alex. Nice to meet you, Alex! Is there anything specific you'd like to talk about or any questions you have?"

**我们携带了历史消息后，模型知道我们的名字了**。
> 但是如果我们每次会话，都需要手动携带历史会话内容，那有点麻烦。

### 3. 自动携带历史会话

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [8]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 携带历史消息的记录
with_message_history = RunnableWithMessageHistory(chain, get_session_history=get_session_history)

In [9]:
# 会话先来个配置，主要配置session_id
config = {
    "configurable": {"session_id": "123"}
}

# 现在开始会话
with_message_history.invoke(
    [HumanMessage(content="Hi! I am alex.")],
    config=config
)


"Hello Alex! Nice to meet you. How can I assist you today? Whether it's information you're looking for or help with something specific, feel free to let me know!"

In [10]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

'Your name is Alex!'

**现在，如果我们把session_id改一下，看还能知道我们名字不？**

In [11]:
# 另外一个seesion_id
config2 = {
    "configurable": {"session_id": "126"}
}
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config2
)

"You mentioned that you want to know your own name, but since I don't have access to external information about you, I can't directly tell you what your name is. If you tell me your name, I'd be happy to welcome you by it! How can I assist you today?"

> 换了个新的session id就不知道我的名字了。

In [12]:
# 给新的seesion id会话中，告诉模型名字
with_message_history.invoke(
    [HumanMessage(content="Hi, I am codelieche")],
    config=config2
)

'Hello codelieche! Nice to meet you. How can I assist you today? Whether you have any questions, need some help with coding, or just want to chat, feel free to let me know!'

In [13]:
# 再次询问名字
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config2
)

'Your name is codelieche. How can I assist you further, codelieche?'

In [14]:
store["126"].messages

[HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="You mentioned that you want to know your own name, but since I don't have access to external information about you, I can't directly tell you what your name is. If you tell me your name, I'd be happy to welcome you by it! How can I assist you today?", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi, I am codelieche', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello codelieche! Nice to meet you. How can I assist you today? Whether you have any questions, need some help with coding, or just want to chat, feel free to let me know!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your name is codelieche. How can I assist you further, codelieche?', additional_kwargs={}, response_metadata={})]

#### 4. 加入提示词模板

In [15]:
system_message_prompt = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
    # 输入的字段
    input_variables=["language"],
    # 设置字段默认的值
    partial_variables={"language": "english"}
)
prompt_template = ChatPromptTemplate.from_messages([
    # SystemMessage(content="You are a helpful assistant. Answer all questions to the best of your ability in {language}."),
    system_message_prompt,
    MessagesPlaceholder(variable_name="messages"),
    
])

In [16]:
chain2 = prompt_template | model | str_parser

In [17]:
chain2.invoke({"messages": [HumanMessage(content="Hi, I am alex.")]})

"Hello Alex! Nice to meet you. How can I assist you today? Is there anything specific you'd like to discuss or any questions you have?"

In [18]:
chain2.invoke({"messages": [HumanMessage(content="What's my name?")], "language": "Chinese"})

'抱歉，您没有提供您的名字，我无法回答您的问题。您可以告诉我您的名字吗？'

In [19]:
# 使用记忆： 如果是要传递多个input的key的，那么需要指定input_messages_key的值
with_msg_history = RunnableWithMessageHistory(chain2, get_session_history=get_session_history, input_messages_key="messages")

In [20]:
# 定义配置
config3 = {
    "configurable": {"session_id": "abc"}
}

In [21]:
# 直接询问我的名字
with_msg_history.invoke({
    "messages": [HumanMessage(content="What's my name?")], 
    "language": "chinese"
}, config=config3)

'很抱歉，我无法知道您的名字，因为我们初次见面，我没有相关的信息。您可以告诉我您叫什么名字吗？'

In [22]:
# 告知我的名字
with_msg_history.invoke({
    "messages": [HumanMessage(content="Hi, I am alex.")]
}, config=config3)

'Hello Alex! Nice to meet you. How can I assist you today?'

In [23]:
# 再次询问：我的名字
with_msg_history.invoke({
    "messages": [HumanMessage(content="What's my name?")]
}, config=config3)

'Your name is Alex.'

**现在引出一个问题：** 如果我会话长度有100次，10000次，那么全部消息都记录，太占资源了。这个时候，需要截取一下消息长度

### 5. 管理对话历史：截取次数

In [24]:
from langchain_core.runnables import RunnablePassthrough

In [25]:
def filter_messages(messages, num=5):
    # 只获取最近num次的消息
    return messages[-num:]

# 定义新的链
chain5 = (
    RunnablePassthrough.assign(messages=lambda x : filter_messages(x["messages"]))
    | prompt_template
    | model
    | str_parser               
)

In [26]:
# 验证一下： 我设置四次消息
messages = [
    HumanMessage(content="Hi~ I am alex"),
    AIMessage(content="Hello Alex! Nice to meet you."),
    HumanMessage(content="I like ice cream very mutch."),
    AIMessage(content="Icecream is very good."),
    HumanMessage(content="What's my name?")
]

In [27]:
chain5.invoke({"messages": messages})

'Your name is Alex.'

In [28]:
# 验证一下： 我设置四次消息
messages = [
    HumanMessage(content="Hi~ I am alex"),
    AIMessage(content="Hello Alex! Nice to meet you."),
    HumanMessage(content="I like ice cream very mutch."),
    AIMessage(content="Ice  cream is very good."),
    HumanMessage(content="What's my name?")
]

In [29]:
chain5.invoke({"messages": messages})

'Your name is Alex.'

#### 历史会话

In [30]:
with_messages_history_05 = RunnableWithMessageHistory(
    chain5,
    get_session_history=get_session_history,
    input_messages_key="messages"
)
config_05 = {"configurable": {"session_id": "abcdefg"}}

In [31]:
with_messages_history_05.invoke({"messages": messages}, config=config_05)

'Your name is Alex.'

In [32]:
with_messages_history_05.invoke({"messages": [HumanMessage(content="What is 1 + 1?")]}, config=config_05)

'1 + 1 equals 2.'

In [33]:
with_messages_history_05.invoke({"messages": [HumanMessage(content="What is 1 + 101?")], "language": "chinese"}, config=config_05)

'1 + 101 等于 102。'

In [34]:
with_messages_history_05.invoke({"messages": [HumanMessage(content="What is 202 * 101?")], "language": "chinese"}, config=config_05)

'202 乘以 101 等于 20402。'

In [35]:
# 现在我们可以知道会话聊天的消息长度已经大于5条了，再问名字，估计不知道我是谁了
with_messages_history_05.invoke({"messages": [HumanMessage(content="What is my name?")], "language": "Chinese"}, config=config_05)

'抱歉，您没有提供您的名字，所以我无法回答您的问题。您可以告诉我您的名字是什么吗？'

In [36]:
# 流式响应
resp = with_messages_history_05.stream({"messages": [HumanMessage(content="What is my name?")], "language": "Chinese"}, config=config_05)
resp

<generator object RunnableBindingBase.stream at 0x111a4c840>

In [37]:
for r in resp:
    print(r, end=" ")

很 抱歉 ， 我 无法 知道 您的 姓名 ， 因为我 没有 其他 信息 来源 。 如果您 愿意 告知 ， 我很 乐意 知道 您的 名字 。 您的 名字 是什么 ？  